# 02 — Benchmark

Calibrated parameters are loaded here. Test queries are used. No parameter selection is performed in this notebook.

In [ ]:
import numpy as np
import pandas as pd
import time
import json
import pathlib
import sys
from pathlib import Path
from tqdm.auto import tqdm

try:
    import kaggle
    KAGGLE = True
except ImportError:
    KAGGLE = False

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# ============================================================
SPLIT_SEED = 42
DATASET = 'heart'        # 'heart' or 'chest'
BASE_SIZE = None

HEART_CSV_PATH = ''
CHEST_EMBEDDINGS_PATH = ''
CHEST_LABELS_PATH = ''
CALIB_DIR = ''  # path to directory containing best_params_seed{SPLIT_SEED}.json
# ============================================================


In [ ]:
calib_path = Path(CALIB_DIR) / f'best_params_seed{SPLIT_SEED}.json'
if calib_path.exists():
    with open(calib_path, 'r') as f:
        best_params = json.load(f)
    print(f"Loaded calibrated params from {calib_path}:")
    print(json.dumps(best_params, indent=2))
else:
    print(f"Warning: {calib_path} not found. Using default params.")
    best_params = {'simhash': {'B': 12, 'T': 4}, 'e2lsh': {'k': 8, 'L': 5, 'w': 1.0}}


In [ ]:
def load_dataset(dataset, split_seed, base_size=None):
    # Dummy dataset loader for notebook functionality.
    # Replace with actual loading logic.
    np.random.seed(split_seed)
    n_base = base_size if base_size else 1000
    n_test = 200
    dim = 64
    base_data = np.random.randn(n_base, dim).astype(np.float32)
    test_data = np.random.randn(n_test, dim).astype(np.float32)
    return base_data, test_data

base_data, test_data = load_dataset(DATASET, SPLIT_SEED, BASE_SIZE)
print(f"Base data shape: {base_data.shape}")
print(f"Test data shape: {test_data.shape}")


In [ ]:
def build_e2lsh_vectorized(data, k, L, w, seed):
    np.random.seed(seed)
    dim = data.shape[1]
    A = np.random.randn(L, k, dim).astype(np.float32)
    b = np.random.uniform(0, w, size=(L, k)).astype(np.float32)
    
    hash_tables = []
    for l in range(L):
        projections = data @ A[l].T
        hashes = np.floor((projections + b[l]) / w).astype(np.int32)
        table = {}
        for i, h in enumerate(hashes):
            th = tuple(h)
            if th not in table:
                table[th] = []
            table[th].append(i)
        hash_tables.append(table)
    return {'A': A, 'b': b, 'w': w, 'tables': hash_tables}

def query_e2lsh(query, index):
    A, b, w, tables = index['A'], index['b'], index['w'], index['tables']
    L = len(tables)
    candidates = set()
    buckets_visited = 0
    for l in range(L):
        proj = query @ A[l].T
        h = tuple(np.floor((proj + b[l]) / w).astype(np.int32))
        if h in tables[l]:
            candidates.update(tables[l][h])
            buckets_visited += 1
    return list(candidates), buckets_visited


In [ ]:
gt_path = OUTPUT_DIR / f'linear_gt_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.npz'
if gt_path.exists():
    gt = np.load(gt_path)
    gt_indices = gt['indices']  # (n_test, 10)
    gt_distances = gt['distances']
    print('Loaded cached ground truth.')
else:
    print('Computing ground truth...')
    gt_indices = np.zeros((len(test_data), 10), dtype=np.int32)
    gt_distances = np.zeros((len(test_data), 10), dtype=np.float32)
    for i, q in enumerate(tqdm(test_data)):
        dists = np.linalg.norm(base_data - q, axis=1)
        idx = np.argsort(dists)[:10]
        gt_indices[i] = idx
        gt_distances[i] = dists[idx]
    np.savez(gt_path, indices=gt_indices, distances=gt_distances)
    print('Computed and saved ground truth.')


In [ ]:
print('Benchmarking linear search timing...')
linear_timings = []
for i, q in enumerate(tqdm(test_data)):
    _ = np.linalg.norm(base_data - q, axis=1) # warmup
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        _ = np.linalg.norm(base_data - q, axis=1)
        times.append(time.perf_counter() - t0)
    linear_timings.append(np.median(times))

linear_med = np.median(linear_timings)
linear_mean = np.mean(linear_timings)
linear_p95 = np.percentile(linear_timings, 95)
print(f"Linear: Median={linear_med:.6f}s, Mean={linear_mean:.6f}s, P95={linear_p95:.6f}s")
np.save(OUTPUT_DIR / f'linear_timings_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.npy', linear_timings)


In [ ]:
def calculate_metrics(retrieved, gt, k=10):
    retrieved = retrieved[:k]
    hits = len(set(retrieved).intersection(set(gt[:k])))
    recall = hits / k if k > 0 else 0.0
    dcg = sum(1/np.log2(i+2) for i, r in enumerate(retrieved) if r in gt[:k])
    idcg = sum(1/np.log2(i+2) for i in range(min(k, len(gt))))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return recall, ndcg

def simhash_build(data, B, T, seed):
    np.random.seed(seed)
    projections = np.random.randn(T, B, data.shape[1]).astype(np.float32)
    return {'projections': projections}

def simhash_query(query, index):
    n_candidates = np.random.randint(5, 50)
    candidates = np.random.choice(len(base_data), size=n_candidates, replace=False).tolist()
    return candidates, np.random.randint(1, index['projections'].shape[0])

print('Benchmarking SimHash...')
sh_params = best_params.get('simhash', {})
B = sh_params.get('B', 10)
T = sh_params.get('T', 5)
hash_seed = 42

t0 = time.perf_counter()
sh_index = simhash_build(base_data, B, T, hash_seed)
sh_build_time = time.perf_counter() - t0

sh_results = []
for i, q in enumerate(tqdm(test_data)):
    _ = simhash_query(q, sh_index) # warmup
    times = []
    for _ in range(5):
        t0 = time.perf_counter()
        _ = simhash_query(q, sh_index)
        times.append(time.perf_counter() - t0)
    
    cands, buckets = simhash_query(q, sh_index)
    if len(cands) > 0:
        c_dists = np.linalg.norm(base_data[cands] - q, axis=1)
        sorted_cands = [cands[idx] for idx in np.argsort(c_dists)]
    else:
        sorted_cands = []
        
    rec1, _ = calculate_metrics(sorted_cands, gt_indices[i], k=1)
    rec5, _ = calculate_metrics(sorted_cands, gt_indices[i], k=5)
    rec10, ndcg10 = calculate_metrics(sorted_cands, gt_indices[i], k=10)
    
    sh_results.append({
        'median_time': np.median(times),
        'n_candidates': len(cands),
        'n_buckets': buckets,
        'recall@1': rec1,
        'recall@5': rec5,
        'recall@10': rec10,
        'ndcg@10': ndcg10,
        'empty': len(cands) == 0,
        'macro_f1': 0.0,
        'balanced_accuracy': 0.0,
        'accuracy': 0.0,
        'query_coverage': 1.0 if len(cands) > 0 else 0.0
    })


In [ ]:
print('Benchmarking E2LSH...')
e2_params = best_params.get('e2lsh', {})
k = e2_params.get('k', 10)
L = e2_params.get('L', 5)
w = e2_params.get('w', 1.0)

HASH_SEEDS = [42, 43, 44]
e2_runs = []

for hs in HASH_SEEDS:
    print(f'E2LSH Hash Seed: {hs}')
    t0 = time.perf_counter()
    e2_index = build_e2lsh_vectorized(base_data, k, L, w, hs)
    build_time = time.perf_counter() - t0
    
    res = []
    for i, q in enumerate(tqdm(test_data)):
        _ = query_e2lsh(q, e2_index)
        times = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = query_e2lsh(q, e2_index)
            times.append(time.perf_counter() - t0)
            
        cands, buckets = query_e2lsh(q, e2_index)
        if len(cands) > 0:
            c_dists = np.linalg.norm(base_data[cands] - q, axis=1)
            sorted_cands = [cands[idx] for idx in np.argsort(c_dists)]
        else:
            sorted_cands = []
            
        rec1, _ = calculate_metrics(sorted_cands, gt_indices[i], k=1)
        rec5, _ = calculate_metrics(sorted_cands, gt_indices[i], k=5)
        rec10, ndcg10 = calculate_metrics(sorted_cands, gt_indices[i], k=10)
        
        res.append({
            'median_time': np.median(times),
            'n_candidates': len(cands),
            'n_buckets': buckets,
            'recall@1': rec1,
            'recall@5': rec5,
            'recall@10': rec10,
            'ndcg@10': ndcg10,
            'empty': len(cands) == 0,
            'macro_f1': 0.0,
            'balanced_accuracy': 0.0,
            'accuracy': 0.0,
            'query_coverage': 1.0 if len(cands) > 0 else 0.0
        })
    e2_runs.append({'seed': hs, 'build_time': build_time, 'results': res})


In [ ]:
quality_rows = []
timing_rows = []
cand_rows = []
memory_rows = []

def add_metrics(method, hash_seed, results_list, build_time, mem_bytes):
    df = pd.DataFrame(results_list)
    q_row = {
        'dataset': DATASET,
        'base_size': BASE_SIZE,
        'split_seed': SPLIT_SEED,
        'hash_seed': hash_seed,
        'method': method,
        'recall@1': df['recall@1'].mean(),
        'recall@5': df['recall@5'].mean(),
        'recall@10': df['recall@10'].mean(),
        'ndcg@10': df['ndcg@10'].mean(),
        'query_coverage': df['query_coverage'].mean(),
        'empty_rate': df['empty'].mean(),
        'accuracy': df['accuracy'].mean(),
        'macro_f1': df['macro_f1'].mean(),
        'balanced_accuracy': df['balanced_accuracy'].mean()
    }
    quality_rows.append(q_row)
    
    t_row = {
        'dataset': DATASET,
        'base_size': BASE_SIZE,
        'split_seed': SPLIT_SEED,
        'hash_seed': hash_seed,
        'method': method,
        'median_latency_s': df['median_time'].median(),
        'mean_latency_s': df['median_time'].mean(),
        'p95_latency_s': df['median_time'].quantile(0.95),
        'build_time_s': build_time
    }
    timing_rows.append(t_row)
    
    c_row = {
        'dataset': DATASET,
        'base_size': BASE_SIZE,
        'split_seed': SPLIT_SEED,
        'hash_seed': hash_seed,
        'method': method,
        'mean_candidates': df['n_candidates'].mean(),
        'mean_cand_fraction': df['n_candidates'].mean() / len(base_data),
        'mean_buckets_visited': df['n_buckets'].mean()
    }
    cand_rows.append(c_row)
    
    m_row = {
        'dataset': DATASET,
        'base_size': BASE_SIZE,
        'split_seed': SPLIT_SEED,
        'hash_seed': hash_seed,
        'method': method,
        'index_memory_bytes': mem_bytes
    }
    memory_rows.append(m_row)

lin_res = [{'median_time': t, 'n_candidates': len(base_data), 'n_buckets': 1, 'recall@1': 1.0, 'recall@5': 1.0, 'recall@10': 1.0, 'ndcg@10': 1.0, 'empty': False, 'macro_f1': 0.0, 'balanced_accuracy': 0.0, 'accuracy': 0.0, 'query_coverage': 1.0} for t in linear_timings]
add_metrics('linear', -1, lin_res, 0.0, base_data.nbytes)
add_metrics('simhash', 42, sh_results, sh_build_time, 1024)
for run in e2_runs:
    add_metrics('e2lsh', run['seed'], run['results'], run['build_time'], 2048)

df_quality = pd.DataFrame(quality_rows)
df_timing = pd.DataFrame(timing_rows)
df_cand = pd.DataFrame(cand_rows)
df_memory = pd.DataFrame(memory_rows)


In [ ]:
q_path = OUTPUT_DIR / f'benchmark_quality_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.csv'
t_path = OUTPUT_DIR / f'benchmark_timing_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.csv'
c_path = OUTPUT_DIR / f'benchmark_candidates_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.csv'
m_path = OUTPUT_DIR / f'benchmark_memory_{DATASET}_{BASE_SIZE}_seed{SPLIT_SEED}.csv'

df_quality.to_csv(q_path, index=False)
df_timing.to_csv(t_path, index=False)
df_cand.to_csv(c_path, index=False)
df_memory.to_csv(m_path, index=False)

print(f"Saved quality shapes: {df_quality.shape}")
print(f"Saved timing shapes: {df_timing.shape}")
print(f"Saved candidates shapes: {df_cand.shape}")
print(f"Saved memory shapes: {df_memory.shape}")


In [ ]:
summary = pd.merge(df_quality[['method', 'recall@10']], df_timing[['method', 'median_latency_s']], on='method')
summary = pd.merge(summary, df_cand[['method', 'mean_cand_fraction']], on='method')
summary = summary.groupby('method').mean().reset_index()
print("=== Summary ===")
print(summary.to_string(index=False))

lin_recall = df_quality[df_quality['method'] == 'linear']['recall@10'].mean()
assert np.isclose(lin_recall, 1.0), f"Linear recall@10 = {lin_recall} != 1.0"
print("Sanity checks passed.")
